<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# EduTrack Analytics
## Notebook 1 — Contexte, Brief Métier & Découverte des Données
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Projet** | EduTrack Analytics |
| **Période** | Janvier 2022 — Juin 2024 |
| **Niveau** | Intermédiaire |
| **Outils** | Python — pandas, matplotlib |
| **Durée estimée** | 2h à 3h |

---
## 0. Mise en place de l'environnement

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/elearning_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

> **MÉTHODE — Pourquoi centraliser les couleurs dans `COLORS` ?**
>
> Le dictionnaire `COLORS` centralise la palette DataProjectLab. En l'utilisant systématiquement, tous les graphiques du projet ont une identité visuelle cohérente. `plt.rcParams` définit un style global propre sans effort supplémentaire à chaque graphique.

---
## Étape 1 — Chargement des 5 fichiers CSV

### MÉTHODE
Le paramètre `parse_dates` est **obligatoire** dès le chargement. Si on charge la colonne en string et qu'on convertit après avec `pd.to_datetime()`, on risque des erreurs silencieuses sur les dates mal formatées. En le faisant au `read_csv`, pandas détecte et signale immédiatement les problèmes.

**Règle :** Ne jamais charger une colonne de date comme string pour la convertir plus tard.

> **Chargement depuis GitHub :** les fichiers bruts sont hébergés sur le repo public DataProjectLab. `BASE_URL` pointe directement vers les CSV — aucune installation supplémentaire, fonctionne en Colab et en local.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/'

# Chargement des 5 fichiers CSV
df_parc  = pd.read_csv(BASE_URL + 'parcours.csv')
df_app   = pd.read_csv(BASE_URL + 'apprenants.csv',
                        parse_dates=['date_inscription'])
df_inscr = pd.read_csv(BASE_URL + 'inscriptions.csv',
                        parse_dates=['date_inscription', 'date_fin_reelle'])
df_sess  = pd.read_csv(BASE_URL + 'sessions.csv',
                        parse_dates=['date_session'])
df_paie  = pd.read_csv(BASE_URL + 'paiements.csv',
                        parse_dates=['date_paiement'])

for name, df in [('parcours', df_parc), ('apprenants', df_app),
                  ('inscriptions', df_inscr), ('sessions', df_sess),
                  ('paiements', df_paie)]:
    print(f'{name:<20} {df.shape[0]:>7,} lignes x {df.shape[1]} colonnes')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 2 — Exploration des tables

### MÉTHODE
`.info()` donne en une seule commande les types, la présence de nulls et la mémoire consommée. C'est le premier réflexe avant toute analyse. On identifie aussi les clés de jointure qui permettront d'assembler les tables dans les notebooks suivants.

In [ ]:
# Exploration complète des tables
for name, df in [('parcours', df_parc), ('apprenants', df_app),
                  ('inscriptions', df_inscr), ('sessions', df_sess),
                  ('paiements', df_paie)]:
    print(f'\n{"="*50}')
    print(f'TABLE : {name.upper()}')
    print(df.info())
    print(df.head(3))

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 3 — Qualité des données

### MÉTHODE
Le diagnostic qualité doit être systématique et documenté. On distingue :
- **Nulls** : données manquantes — à imputer ou à signaler
- **Doublons** : entités dupliquées — à supprimer en gardant la référence la plus fiable
- **Incohérences** : valeurs hors plage (âges négatifs, progressions > 100%) — à corriger avec une règle métier explicite

In [ ]:
# Valeurs nulles par table
print('=== VALEURS NULLES ===')
for name, df in [('apprenants', df_app), ('inscriptions', df_inscr),
                  ('sessions', df_sess), ('paiements', df_paie)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    # TODO : afficher les nulls si > 0, sinon afficher 'aucun null'
    pass

In [ ]:
# TODO : détecter les doublons sur email dans df_app
# TODO : identifier les âges aberrants (< 16 ou > 80)
# TODO : détecter les progressions > 100% dans df_inscr
# TODO : identifier les sessions avec durée <= 0
# TODO : identifier les paiements avec montant <= 0

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 4 — Premiers KPIs EduTrack

### MÉTHODE
On calcule d'abord les KPIs bruts avant tout nettoyage pour avoir une photographie de l'état réel des données. Ces chiffres serviront de référence pour valider que le nettoyage (NB2) n'a pas altéré les ordres de grandeur.

In [ ]:
DATE_REF = pd.Timestamp('2024-06-30')

# TODO : calculer les 5 KPIs
nb_apprenants  = # df_app['apprenant_id'].nunique()
taux_abandon   = # (df_inscr['statut'] == 'Abandonne').mean() * 100
taux_completion= # TODO
revenu_total   = # TODO : sum des paiements valides
csat_moyen     = # TODO : AVG sur statut == 'Termine' uniquement

print('=' * 45)
print(f'{"EduTrack — KPIs Globaux":^45}')
print('=' * 45)
# TODO : afficher les KPIs

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
# TODO : afficher la distribution des statuts en valeur absolue et en %
# Indice : df_inscr['statut'].value_counts()

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 5 — Visualisation 2×2

### MÉTHODE
Une figure 2×2 permet de présenter 4 dimensions complémentaires en une seule image. On utilise `COLORS` pour les couleurs et `tight_layout()` pour éviter les chevauchements entre les sous-graphiques.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("EduTrack — Vue d'ensemble", fontsize=16, fontweight='bold',
             color=COLORS['primary'], y=1.01)

# Graphique 1 — Inscriptions mensuelles
# TODO : calculer monthly (resample mensuel sur date_inscription)
# TODO : tracer la courbe avec axes[0,0]

# Graphique 2 — Inscriptions par domaine
# TODO : joindre df_parc et df_inscr sur parcours_id, compter par domaine
# TODO : tracer les barres horizontales avec axes[0,1]

# Graphique 3 — Statuts
# TODO : compter les statuts et tracer avec axes[1,0]

# Graphique 4 — Top 5 pays
# TODO : compter les pays dans df_app et tracer avec axes[1,1]

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}vue_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Bilan du Notebook 1

| KPI | Valeur |
|---|---|
| Nombre total d'apprenants | **4 500** |
| Total inscriptions | **6 460** |
| Taux d'abandon global | **17.7%** |
| Taux de complétion | **33.3%** |
| Revenu total | **748.7 M FCFA** |
| CSAT moyen | **4.23 / 5** |

**Anomalies détectées :** 3 emails dupliqués, 5 âges aberrants, 2 pays null, 4 progressions > 100%, 6 durées sessions négatives, 3 montants négatifs

**Pour le Notebook 2 :** Corriger toutes les anomalies et créer les features `engagement_score`, `nb_jours_inactif` et `at_risk_dropout`.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.